In [46]:
import os
import sys
word_dir = '/Users/lxx/Documents/codes/st-missing-fill'
os.chdir(word_dir)
sys.path.append(word_dir)


In [47]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pypots.imputation import  LOCF, GRUD, USGAN, SAITS, iTransformer


from src.data.load import load_data
from src.data.misser import mcar_masker, seq_masker, scm_masker, mask_mapping

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
%config InlineBackend.figure_format = 'retina'

Using device: mps


In [49]:
def evaluate_rmse(y_hat, ground_y, mask): # 评估函数
    err = (y_hat - ground_y)
    err = err[mask==0] # 只计算缺失值位置的误差
    return np.sqrt(np.mean(err**2)).round(4) # 保留四位小数

def run_locf_imputation(y_mcar): # 对比模型：LOCF
    model = LOCF(device=device)
    y_hat = model.impute({"X": y_mcar}).squeeze()
    return y_hat


In [50]:
def simulate_missingness(ground_y, pi, miss_pattern, S_cluster=None):
    if miss_pattern == 'mcar':
        mask = mcar_masker(ground_y.shape, pi)
    elif miss_pattern == 'seq':
        mask = seq_masker(ground_y.shape, pi)
    elif miss_pattern == 'scm':
        mask = scm_masker(S_cluster, ground_y.shape, pi)
    else:
        raise ValueError(f'Unknown missing pattern: {miss_pattern}')
    y_pattern = mask_mapping(ground_y, mask)
    y_pattern = y_pattern[..., np.newaxis]
    return y_pattern, mask

In [51]:
ground_X, ground_y, all_stations, all_stations_cluster, vars = load_data()

Missing rate in ground_y: 0.27%
Missing rate in ground_X: 0.18%


In [52]:
res = []
lis_pi = [0.1, 0.3, 0.5]
for pi in lis_pi:
    resi = []
    for pattern in ['mcar', 'seq', 'scm']:
        y_masked, mask = simulate_missingness(ground_y, pi, pattern, all_stations_cluster)
        y_hat = run_locf_imputation(y_masked)
        rmse = evaluate_rmse(y_hat, ground_y, mask)
        resi.append(rmse)
    res.append(resi)
res = pd.DataFrame(res, columns=['MCAR', 'SEQ', 'SCM'], index=lis_pi)
res.index.name = 'Missing Rate'

2026-02-07 22:06:40 [INFO]: Using the given device: mps
2026-02-07 22:06:41 [INFO]: Using the given device: mps
2026-02-07 22:06:41 [INFO]: Using the given device: mps
2026-02-07 22:06:41 [INFO]: Using the given device: mps
2026-02-07 22:06:41 [INFO]: Using the given device: mps
2026-02-07 22:06:42 [INFO]: Using the given device: mps
2026-02-07 22:06:42 [INFO]: Using the given device: mps
2026-02-07 22:06:42 [INFO]: Using the given device: mps
2026-02-07 22:06:43 [INFO]: Using the given device: mps


In [53]:
res

,MCAR,SEQ,SCM
Missing Rate,,,
0.1,3.0816,3.0806,3.0780
0.3,3.2122,3.2218,3.2040
0.5,3.3395,3.3458,3.3375
